# Exercise 11.1: Z → μμ with CMS open data

From *Programming in High Energy Particle Physics*, Chapter 11

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch11/ex11_1_solution.ipynb)

Exercise 11.1 Z → μμ with CMS open data. Use the public [Run2016G DoubleMuon NanoAODv9 record](https://opendata.cern.ch/record/30522) and the matching 2016 [certified-luminosity JSON record](https://opendata.cern.ch/record/14220). The selected file is 66,090,536 bytes. Read at most 200,000 events, apply the Golden JSON, require two tight opposite-charge muons, and fit their invariant-mass peak. If a download fails, an explicitly labelled synthetic fallback keeps the notebook runnable.

<details><summary>Hint</summary>

Read `Muon_pt`, `Muon_eta`, `Muon_phi`, `Muon_charge`, `Muon_tightId`, `run`, and `luminosityBlock` from `Events`. Select certified run/luminosity-section pairs before choosing the two highest-$p_T$ accepted muons. Fit a Voigtian plus background. A fit to one small uncalibrated file is an exercise, not a precision CMS result.

</details>

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q uproot awkward vector scipy matplotlib numpy")

### Data source and fallback

The CMS NanoAOD and Golden JSON are the default. A seeded synthetic sample is used only when either download fails and is identified in the printed source.

### Step 1: Load CMS-like NanoAOD branches and apply a certified mask

In [ ]:
import json, urllib.request
import numpy as np, awkward as ak, uproot

NANOAOD_URL = "https://opendata.cern.ch/eos/opendata/cms/Run2016G/DoubleMuon/NANOAOD/UL2016_MiniAODv2_NanoAODv9-v2/2430000/4E3A26DE-E53B-A844-8048-36376617AE8D.root"
GOLDEN_JSON_URL = "https://opendata.cern.ch/eos/opendata/cms/validated-runs/2016/Cert_271036-284044_13TeV_Legacy2016_Collisions16_JSON.txt"
BRANCHES = ["Muon_pt", "Muon_eta", "Muon_phi", "Muon_charge", "Muon_tightId", "run", "luminosityBlock"]
ENTRY_STOP = 200000
source = "CMS Run2016G DoubleMuon NanoAODv9"
try:
    with urllib.request.urlopen(GOLDEN_JSON_URL, timeout=30) as response:
        golden = json.load(response)
    with uproot.open(NANOAOD_URL, timeout=60) as root_file:
        events = root_file["Events"].arrays(BRANCHES, entry_stop=ENTRY_STOP, library="ak")
except (OSError, TimeoutError, urllib.error.URLError) as exc:
    print(f"CMS/Golden JSON download failed ({exc}); using synthetic fallback, not CERN data.")
    source = "synthetic fallback"
    rng = np.random.default_rng(42)
    n = 20000
    masses_seed = np.clip(91.1876 + rng.standard_cauchy(n) * 1.25 + rng.normal(0, 1.3, n), 65, 120)
    pt = masses_seed / 2
    events = ak.Array({"Muon_pt": np.stack([pt, pt], axis=1),
                       "Muon_eta": np.zeros((n, 2)),
                       "Muon_phi": np.tile([0.0, np.pi], (n, 1)),
                       "Muon_charge": np.tile([1, -1], (n, 1)),
                       "Muon_tightId": np.ones((n, 2), dtype=bool),
                       "run": np.full(n, 1),
                       "luminosityBlock": rng.integers(1, 101, n)})
    golden = {"1": [[1, 100]]}
print("Source:", source, "events read:", len(events))


### Step 2: Select tight opposite-charge muon pairs

In [ ]:
import vector
vector.register_awkward()

def certified_mask(run, luminosity_block, certified):
    return np.fromiter(
        (any(lo <= int(ls) <= hi for lo, hi in certified.get(str(int(r)), []))
         for r, ls in zip(run, luminosity_block)),
        dtype=bool, count=len(run))

events = events[certified_mask(events.run, events.luminosityBlock, golden)]
print("Certified events:", len(events))
muons = ak.zip({"pt": events.Muon_pt, "eta": events.Muon_eta,
                "phi": events.Muon_phi,
                "mass": ak.ones_like(events.Muon_pt) * 0.105658,
                "charge": events.Muon_charge,
                "tight": events.Muon_tightId}, with_name="Momentum4D")
muons = muons[(muons.pt > 25) & (abs(muons.eta) < 2.4) & muons.tight]
muons = muons[ak.num(muons) >= 2]
muons = muons[ak.argsort(muons.pt, axis=1, ascending=False)][:, :2]
muons = muons[muons[:, 0].charge * muons[:, 1].charge < 0]
masses = ak.to_numpy((muons[:, 0] + muons[:, 1]).mass)
masses = masses[(masses > 70) & (masses < 110)]
print("Selected Z candidates:", len(masses))
assert len(masses) > 100


### Step 3: Fit a Voigtian and estimate uncertainty

In [ ]:
from scipy.optimize import curve_fit
from scipy.special import voigt_profile
import matplotlib.pyplot as plt

bins = np.linspace(70, 110, 81)
counts, _ = np.histogram(masses, bins)
centers = (bins[1:] + bins[:-1]) / 2
bin_width = bins[1] - bins[0]

def model(x, amplitude, mass, resolution, background):
    return amplitude * voigt_profile(x - mass, resolution, 1.2475) * bin_width + background

weights = np.sqrt(np.maximum(counts, 1))
limits = ([0, 85, 0.2, 0], [np.inf, 98, 8, np.inf])
fit, covariance = curve_fit(model, centers, counts, p0=[len(masses), 91.2, 2, 3],
                            sigma=weights, absolute_sigma=True, bounds=limits)
stat = np.sqrt(covariance[1, 1])
core = (centers > 75) & (centers < 105)
fit_core, _ = curve_fit(model, centers[core], counts[core], p0=fit,
                        sigma=weights[core], absolute_sigma=True, bounds=limits)
scale_shift = fit[1] * 0.001  # illustrative 0.1% shift, not a CMS calibration uncertainty
print(f"Measured Z peak: {fit[1]:.4f} ± {stat:.4f} GeV (fit statistical uncertainty)")
print(f"Selected events: {len(masses)}")
print(f"Illustrative 0.1% momentum-scale shift: {scale_shift:.3f} GeV; fit-range shift: {fit_core[1]-fit[1]:+.3f} GeV")
fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(centers, counts, weights, fmt=".", label=f"{source} selected events")
ax.plot(centers, model(centers, *fit), label="Voigt + flat background")
ax.axvline(91.1876, color="black", ls="--", label="reference 91.1876 GeV")
ax.set(xlabel=r"$m_{\mu\mu}$ [GeV]", ylabel="Events / 0.5 GeV")
ax.legend(fontsize=8)
plt.show()


The reference value is $m_Z = 91.1876$ GeV. The quoted statistical uncertainty is from this simple binned Voigt fit to one small file. The 0.1% scale variation is illustrative, not a CMS systematic uncertainty. A calibrated measurement needs the official muon corrections, efficiency treatment, and systematic variations.

### Interpretation

Compare the fitted peak with 91.1876 GeV. This exercise uses one Run2016G DoubleMuon file and the 2016 certified-luminosity JSON. A small uncalibrated sample and a simple fit do not establish a precision CMS Z-mass measurement.